In [2]:
import psycopg2
import csv
import os
from datetime import time

In [32]:
import psycopg2

conn = psycopg2.connect(
    host="localhost",
    port='5432',
    dbname="abcmart-test",
    user="postgres",
    password="1234"             
)


cur = conn.cursor()

In [34]:


sql = """

CREATE TABLE Employees (
    employee_id SERIAL PRIMARY KEY,
    first_name VARCHAR(100) NOT NULL,
    last_name VARCHAR(100) NOT NULL,
    social_security_number CHAR(11) UNIQUE,
    address_line VARCHAR(255),
    city VARCHAR(100),
    state VARCHAR(50),
    zip_code VARCHAR(10),
    date_of_birth DATE,
    email VARCHAR(255) UNIQUE,
    phone_number VARCHAR(50)
);

CREATE TABLE Human_Resources (
    hr_record_id SERIAL PRIMARY KEY,
    employee_id INT NOT NULL,
    conduct_notes TEXT,
    tenure_start_date DATE,
    tenure_end_date DATE,
    promotion_date DATE,
    performance_rating INT CHECK (performance_rating BETWEEN 1 AND 5),
    hr_issue_flag BOOLEAN DEFAULT FALSE,
    FOREIGN KEY (employee_id) REFERENCES Employees(employee_id)
);

CREATE TABLE Customer_Loyalty (
    customer_id SERIAL PRIMARY KEY,
    customer_name VARCHAR(100),
    email VARCHAR(100) UNIQUE,
    phone_number VARCHAR(50),
    loyalty_points INT DEFAULT 0,
    membership_start DATE,
    last_transaction_date DATE,
    reward_tier VARCHAR(50),
    is_active BOOLEAN DEFAULT TRUE
);

CREATE TABLE Vendors (
    vendor_id SERIAL PRIMARY KEY,
    vendor_name VARCHAR(255) NOT NULL,
    contact_phone VARCHAR(50) NOT NULL,
    contact_email VARCHAR(255) NOT NULL,
    address_line1 VARCHAR(255) NOT NULL,
    address_line2 VARCHAR(255),
    city VARCHAR(100) NOT NULL,
    state VARCHAR(50) NOT NULL,
    zip_code VARCHAR(10) NOT NULL,
    country VARCHAR(100) NOT NULL,
    primary_contact_name VARCHAR(255) NOT NULL,
    primary_contact_email VARCHAR(255) NOT NULL,
    primary_contact_phone VARCHAR(50) NOT NULL
);

CREATE TABLE Stores (
    store_id SERIAL PRIMARY KEY,
    store_name VARCHAR(255) NOT NULL,
    address_line1 VARCHAR(255) NOT NULL,
    address_line2 VARCHAR(255),
    city VARCHAR(100) NOT NULL,
    state VARCHAR(50) NOT NULL,
    zip_code VARCHAR(10) NOT NULL,
    country VARCHAR(100) NOT NULL,
    capacity INT NOT NULL,
    hours_of_operation VARCHAR(255) NOT NULL,
    phone_number VARCHAR(50) NOT NULL,
    email VARCHAR(255) NOT NULL,
    manager_name VARCHAR(255) NOT NULL,
    opened_date DATE NOT NULL
);

CREATE TABLE Delivery_Tracking (
    delivery_id SERIAL PRIMARY KEY,
    delivery_date DATE NOT NULL,
    vendor_id INT,
    store_location_id INT,
    quantity_delivered INT,
    brand_name VARCHAR(100),
    delivery_status VARCHAR(50),
    received_by INT,
    FOREIGN KEY (vendor_id) REFERENCES Vendors(vendor_id),
    FOREIGN KEY (store_location_id) REFERENCES Stores(store_id),
    FOREIGN KEY (received_by) REFERENCES Employees(employee_id)


);

CREATE TABLE Inventory (
    inventory_id SERIAL PRIMARY KEY,
    item_name VARCHAR(255) NOT NULL,
    item_description TEXT NOT NULL,
    shelf_date DATE NOT NULL,
    sale_date DATE,
    quantity_in_store INT NOT NULL,
    price DECIMAL(10, 2) NOT NULL,
    upcharge_percentage DECIMAL(5, 2) NOT NULL,
    expiration_date DATE,
    location VARCHAR(50) NOT NULL,
    store_id INT NOT NULL,
    vendor_id INT NOT NULL,
    FOREIGN KEY (store_id) REFERENCES Stores(store_id),
    FOREIGN KEY (vendor_id) REFERENCES Vendors(vendor_id)
);

CREATE TABLE Sales_Management (
    sale_id SERIAL PRIMARY KEY,
    sale_date DATE NOT NULL,
    item_id INT NOT NULL,
    quantity_sold INT NOT NULL,
    discount_applied DECIMAL(5,2),
    sale_amount DECIMAL(10,2),
    store_location_id INT,
    customer_id INT,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    FOREIGN KEY (item_id) REFERENCES Inventory(inventory_id),
    FOREIGN KEY (store_location_id) REFERENCES Stores(store_id),
    FOREIGN KEY (customer_id) REFERENCES Customer_Loyalty(customer_id)
);

CREATE TABLE Returns_Refunds (
    return_id SERIAL PRIMARY KEY,
    sale_id INT NOT NULL,
    return_date DATE NOT NULL,
    item_id INT NOT NULL,
    quantity_returned INT NOT NULL,
    reason_for_return TEXT,
    refund_amount DECIMAL(10,2),
    processed_by INT,
    affects_accounting BOOLEAN DEFAULT TRUE,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    FOREIGN KEY (sale_id) REFERENCES Sales_Management(sale_id),
    FOREIGN KEY (item_id) REFERENCES Inventory(inventory_id),
    FOREIGN KEY (processed_by) REFERENCES Employees(employee_id)
);

CREATE TABLE Operations_Billing (
    billing_id SERIAL PRIMARY KEY,
    bill_type VARCHAR(50) NOT NULL,
    amount DECIMAL(12,2) NOT NULL,
    payment_date DATE,
    invoice_document TEXT,
    status VARCHAR(50),
    store_location_id INT,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    FOREIGN KEY (store_location_id) REFERENCES Stores(store_id)
);

CREATE TABLE Asset_Management (
    asset_id SERIAL PRIMARY KEY,
    asset_type VARCHAR(100) NOT NULL,
    purchase_date DATE,
    asset_value DECIMAL(12,2),
    maintenance_history TEXT,
    store_location_id INT,
    is_active BOOLEAN DEFAULT TRUE,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    FOREIGN KEY (store_location_id) REFERENCES Stores(store_id)
);

CREATE TABLE Vehicle_Management (
    vehicle_id SERIAL PRIMARY KEY,
    vehicle_type VARCHAR(50),
    license_plate VARCHAR(50) UNIQUE NOT NULL,
    assigned_store INT,
    driver_assigned INT,
    lease_status VARCHAR(50),
    last_service_date DATE,
    is_active BOOLEAN DEFAULT TRUE,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    FOREIGN KEY (assigned_store) REFERENCES Stores(store_id),
    FOREIGN KEY (driver_assigned) REFERENCES Employees(employee_id)
);

CREATE TABLE Security_Incidents (
    incident_id SERIAL PRIMARY KEY,
    incident_type VARCHAR(100) NOT NULL,
    incident_date DATE NOT NULL,
    store_location_id INT,
    outcome TEXT,
    reported_by INT,
    resolution_status VARCHAR(50),
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    FOREIGN KEY (store_location_id) REFERENCES Stores(store_id),
    FOREIGN KEY (reported_by) REFERENCES Employees(employee_id)
);

CREATE TABLE Real_Estate_Development (
    project_id SERIAL PRIMARY KEY,
    project_name VARCHAR(255) NOT NULL,
    development_cost DECIMAL(10, 2) NOT NULL,
    project_lead VARCHAR(255) NOT NULL,
    address_line1 VARCHAR(255) NOT NULL,
    address_line2 VARCHAR(255),
    city VARCHAR(100) NOT NULL,
    state VARCHAR(50) NOT NULL,
    zip_code VARCHAR(10) NOT NULL,
    country VARCHAR(100) NOT NULL,
    square_footage INT NOT NULL,
    capacity INT NOT NULL,
    status VARCHAR(50) NOT NULL,
    expected_completion_date DATE NOT NULL
);

CREATE TABLE Transactions (
    transaction_id SERIAL PRIMARY KEY,
    inventory_id INT NOT NULL,
    store_id INT NOT NULL,
    quantity INT NOT NULL,
    transaction_price DECIMAL(10, 2) NOT NULL,
    transaction_date TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
    FOREIGN KEY (inventory_id) REFERENCES Inventory(inventory_id),
    FOREIGN KEY (store_id) REFERENCES Stores(store_id)
);

CREATE TABLE Revenue (
    revenue_id SERIAL PRIMARY KEY,
    period_type VARCHAR(50) NOT NULL CHECK (period_type IN ('Weekly', 'Monthly', 'Yearly')),
    period_start_date DATE NOT NULL,
    period_end_date DATE NOT NULL,
    store_id INT NOT NULL,
    total_revenue DECIMAL(15, 2) NOT NULL,
    FOREIGN KEY (store_id) REFERENCES Stores(store_id)
);

CREATE TABLE Cost_Of_Goods (
    cog_id SERIAL PRIMARY KEY,
    inventory_id INT NOT NULL,
    quantity_sold INT NOT NULL,
    unit_cost DECIMAL(10, 2) NOT NULL,
    total_cost DECIMAL(15, 2) GENERATED ALWAYS AS (quantity_sold * unit_cost) STORED,
    period_type VARCHAR(50) NOT NULL CHECK (period_type IN ('Weekly', 'Monthly', 'Yearly')),
    period_start_date DATE NOT NULL,
    period_end_date DATE NOT NULL,
    store_id INT NOT NULL,
    FOREIGN KEY (inventory_id) REFERENCES Inventory(inventory_id),
    FOREIGN KEY (store_id) REFERENCES Stores(store_id)
);


CREATE TABLE Promotions (
    promotion_id SERIAL PRIMARY KEY,
    promotion_name VARCHAR(100) NOT NULL,
    start_date DATE NOT NULL,
    end_date DATE,
    discount_percent DECIMAL(5,2),
    applicable_category VARCHAR(50),
    is_active BOOLEAN DEFAULT TRUE,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    notes TEXT);


"""


cur.execute(sql)
conn.commit()
print("Tables created")





Tables created


In [ ]:
import psycopg2
from faker import Faker
import random
from datetime import datetime, timedelta

fake = Faker()

conn = psycopg2.connect(
    host="localhost",
    port='5432',
    dbname="abcmart-test",
    user="postgres",
    password="1234"             
)

cur = conn.cursor()


# 1. Employees
def create_employees(n=1000):
    for _ in range(n):
        cur.execute("""
            INSERT INTO Employees (first_name, last_name, social_security_number, address_line, city, state, zip_code, date_of_birth, email, phone_number)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        """, (
            fake.first_name(), fake.last_name(), fake.ssn(), fake.street_address(), fake.city(), fake.state_abbr(),
            fake.zipcode(), fake.date_of_birth(), fake.unique.email(), fake.phone_number()
        ))

# 2. Human_Resources
def create_human_resources(n=1000):
    for i in range(1, n + 1):
        cur.execute("""
            INSERT INTO Human_Resources (employee_id, conduct_notes, tenure_start_date, tenure_end_date, promotion_date, performance_rating, hr_issue_flag)
            VALUES (%s, %s, %s, %s, %s, %s, %s)
        """, (
            i, fake.text(), fake.date_between('-5y', '-2y'), fake.date_between('-1y', 'today'), fake.date_between('-3y', 'today'),
            random.randint(1, 5), fake.boolean()
        ))

# 3. Customer_Loyalty
def create_customer_loyalty(n=1000):
    fake_unique = Faker()
    fake_unique.seed_instance(123)
    for _ in range(n):
        email = fake_unique.unique.email()
        cur.execute("""
            INSERT INTO Customer_Loyalty (customer_name, email, phone_number, loyalty_points, membership_start, last_transaction_date, reward_tier, is_active)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
        """, (
            fake.name(), email, fake.phone_number(), random.randint(0, 1000),
            fake.date_between('-2y', '-1y'), fake.date_between('-30d', 'today'),
            random.choice(['Bronze', 'Silver', 'Gold', 'Platinum']), fake.boolean()
        ))

# 4. Vendors
def create_vendors(n=200):
    for _ in range(n):
        cur.execute("""
            INSERT INTO Vendors (vendor_name, contact_phone, contact_email, address_line1, address_line2, city, state, zip_code, country, primary_contact_name, primary_contact_email, primary_contact_phone)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        """, (
            fake.company(), fake.phone_number(), fake.unique.email(), fake.street_address(), fake.secondary_address(),
            fake.city(), fake.state_abbr(), fake.zipcode(), "USA", fake.name(), fake.email(), fake.phone_number()
        ))

# 5. Stores
def create_stores(n=50):
    for _ in range(n):
        cur.execute("""
            INSERT INTO Stores (store_name, address_line1, address_line2, city, state, zip_code, country, capacity, hours_of_operation, phone_number, email, manager_name, opened_date)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        """, (
            fake.company(), fake.street_address(), fake.secondary_address(), fake.city(), fake.state_abbr(), fake.zipcode(),
            "USA", random.randint(50, 200), "9AM-9PM", fake.phone_number(), fake.unique.email(), fake.name(), fake.date_between('-5y', 'today')
        ))

# 6. Inventory
def create_inventory(n=2000):
    for _ in range(n):
        cur.execute("""
            INSERT INTO Inventory (item_name, item_description, shelf_date, sale_date, quantity_in_store, price, upcharge_percentage, expiration_date, location, store_id, vendor_id)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        """, (
            fake.word(), fake.text(), fake.date_between('-1y', '-30d'), fake.date_between('-30d', 'today'),
            random.randint(1, 100), round(random.uniform(5.0, 100.0), 2), round(random.uniform(5.0, 30.0), 2),
            fake.date_between('today', '+180d'), fake.word(), random.randint(1, 50), random.randint(1, 200)
        ))

# 7. Delivery_Tracking
def create_delivery_tracking(n=500):
    for _ in range(n):
        cur.execute("""
            INSERT INTO Delivery_Tracking (delivery_date, vendor_id, store_location_id, quantity_delivered, brand_name, delivery_status, received_by)
            VALUES (%s, %s, %s, %s, %s, %s, %s)
        """, (
            fake.date_between('-30d', 'today'), random.randint(1, 200), random.randint(1, 50), random.randint(10, 100),
            fake.company(), random.choice(['Delivered', 'Pending', 'Delayed']), random.randint(1, 1000)
        ))

# 8. Sales_Management
def create_sales_management(n=1000):
    for _ in range(n):
        cur.execute("""
            INSERT INTO Sales_Management (sale_date, item_id, quantity_sold, discount_applied, sale_amount, store_location_id, customer_id)
            VALUES (%s, %s, %s, %s, %s, %s, %s)
        """, (
            fake.date_between('-30d', 'today'), random.randint(1, 2000), random.randint(1, 10),
            round(random.uniform(0, 20), 2), round(random.uniform(10, 100), 2),
            random.randint(1, 50), random.randint(1, 1000)
        ))

# 9. Returns_Refunds
def create_returns_refunds(n=300):
    for _ in range(n):
        cur.execute("""
            INSERT INTO Returns_Refunds (sale_id, return_date, item_id, quantity_returned, reason_for_return, refund_amount, processed_by)
            VALUES (%s, %s, %s, %s, %s, %s, %s)
        """, (
            random.randint(1, 1000), fake.date_between('-15d', 'today'), random.randint(1, 2000), random.randint(1, 3),
            fake.sentence(), round(random.uniform(5, 50), 2), random.randint(1, 1000)
        ))

# 10. Operations_Billing
def create_operations_billing(n=300):
    for _ in range(n):
        cur.execute("""
            INSERT INTO Operations_Billing (bill_type, amount, payment_date, invoice_document, status, store_location_id)
            VALUES (%s, %s, %s, %s, %s, %s)
        """, (
            fake.word(), round(random.uniform(100, 1000), 2), fake.date_between('-30d', 'today'), fake.text(),
            random.choice(['Paid', 'Unpaid', 'Pending']), random.randint(1, 50)
        ))

# 11. Asset_Management
def create_asset_management(n=300):
    for _ in range(n):
        cur.execute("""
            INSERT INTO Asset_Management (asset_type, purchase_date, asset_value, maintenance_history, store_location_id)
            VALUES (%s, %s, %s, %s, %s)
        """, (
            fake.word(), fake.date_between('-5y', '-1y'), round(random.uniform(500, 10000), 2), fake.text(), random.randint(1, 50)
        ))

# 12. Vehicle_Management
def create_vehicle_management(n=300):
    for _ in range(n):
        cur.execute("""
            INSERT INTO Vehicle_Management (vehicle_type, license_plate, assigned_store, driver_assigned, lease_status, last_service_date)
            VALUES (%s, %s, %s, %s, %s, %s)
        """, (
            fake.word(), fake.license_plate(), random.randint(1, 50), random.randint(1, 1000), random.choice(['Leased', 'Owned']),
            fake.date_between('-1y', 'today')
        ))

# 13. Security_Incidents
def create_security_incidents(n=300):
    for _ in range(n):
        cur.execute("""
            INSERT INTO Security_Incidents (incident_type, incident_date, store_location_id, outcome, reported_by, resolution_status)
            VALUES (%s, %s, %s, %s, %s, %s)
        """, (
            fake.word(), fake.date_between('-60d', 'today'), random.randint(1, 50), fake.sentence(),
            random.randint(1, 1000), random.choice(['Resolved', 'Pending'])
        ))

# 14. Real_Estate_Development
def create_real_estate_development(n=30):
    for _ in range(n):
        cur.execute("""
            INSERT INTO Real_Estate_Development (project_name, development_cost, project_lead, address_line1, address_line2, city, state, zip_code, country, square_footage, capacity, status, expected_completion_date)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        """, (
            fake.bs(), round(random.uniform(100000, 5000000), 2), fake.name(), fake.street_address(), fake.secondary_address(),
            fake.city(), fake.state_abbr(), fake.zipcode(), "USA", random.randint(1000, 10000), random.randint(50, 300),
            random.choice(['Planning', 'In Progress', 'Completed']), fake.date_between('today', '+1y')
        ))

# 15. Transactions
def create_transactions(n=1000):
    for _ in range(n):
        cur.execute("""
            INSERT INTO Transactions (inventory_id, store_id, quantity, transaction_price)
            VALUES (%s, %s, %s, %s)
        """, (
            random.randint(1, 2000), random.randint(1, 50), random.randint(1, 10), round(random.uniform(10, 200), 2)
        ))

# 16. Revenue
def create_revenue(n=300):
    for _ in range(n):
        cur.execute("""
            INSERT INTO Revenue (period_type, period_start_date, period_end_date, store_id, total_revenue)
            VALUES (%s, %s, %s, %s, %s)
        """, (
            'Monthly', fake.date_between('-60d', '-30d'), fake.date_between('-29d', 'today'), random.randint(1, 50), round(random.uniform(10000, 50000), 2)
        ))

# 17. Cost_Of_Goods
def create_cost_of_goods(n=300):
    for _ in range(n):
        quantity = random.randint(10, 100)
        cost = round(random.uniform(1, 10), 2)
        cur.execute("""
            INSERT INTO Cost_Of_Goods (inventory_id, quantity_sold, unit_cost, period_type, period_start_date, period_end_date, store_id)
            VALUES (%s, %s, %s, %s, %s, %s, %s)
        """, (
            random.randint(1, 2000), quantity, cost, 'Monthly', fake.date_between('-60d', '-30d'), fake.date_between('-29d', 'today'), random.randint(1, 50)
        ))

# 18. Promotions
def create_promotions(n=300):
    for _ in range(n):
        start_date = fake.date_between(start_date='-90d', end_date='-30d')
        end_date = fake.date_between(start_date=start_date, end_date='today')
        cur.execute("""
            INSERT INTO Promotions (promotion_name, start_date, end_date, discount_percent, applicable_category, is_active, notes)
            VALUES (%s, %s, %s, %s, %s, %s, %s)
        """, (
            fake.catch_phrase(),  # promotion_name
            start_date,
            end_date,
            round(random.uniform(5, 50), 2),  # discount_percent
            random.choice(['Electronics', 'Clothing', 'Groceries', 'Toys', 'Home Goods']),
            random.choice([True, False]),
            fake.sentence()
        ))
#19 Staffing
def create_staffing(n=300):
    for _ in range(n):
        shift_start_hour = random.randint(6, 14)  # shifts between 6 AM to 2 PM
        shift_start = time(hour=shift_start_hour, minute=0)
        shift_end = time(hour=(shift_start_hour + 8) % 24, minute=0)
        total_hours = 8.00
        cur.execute("""
            INSERT INTO Staffing (employee_id, schedule_date, shift_start, shift_end, total_hours, store_location_id, is_scheduled)
            VALUES (%s, %s, %s, %s, %s, %s, %s)
        """, (
            random.randint(1, 300),  # assuming you have up to 3000 employees
            fake.date_between(start_date='-60d', end_date='today'),
            shift_start,
            shift_end,
            total_hours,
            random.randint(1, 50),  # store_location_id
            random.choice([True, False])
        ))
# Run all
create_employees()
conn.commit()
create_human_resources()
create_customer_loyalty()
create_vendors()
create_stores()
create_inventory()
create_delivery_tracking()
create_sales_management()
create_returns_refunds()
create_operations_billing()
create_asset_management()
create_vehicle_management()
create_security_incidents()
create_real_estate_development()
create_transactions()
create_revenue()
create_cost_of_goods()
create_promotions()
create_staffing()


# Finalize
conn.commit()
cur.close()
conn.close()

print("Simulated data inserted into all 19 tables!")

In [110]:
import psycopg2


conn = psycopg2.connect(
    host="localhost",
    port='5432',
    dbname="abcmart4",
    user="postgres",
    password="1234"             
)

cur = conn.cursor()

tables = [
    "Employees", "Human_Resources", "Customer_Loyalty", "Vendors", "Stores",
    "Inventory", "Delivery_Tracking", "Sales_Management", "Returns_Refunds",
    "Operations_Billing", "Asset_Management", "Vehicle_Management",
    "Security_Incidents", "Real_Estate_Development", "Transactions",
    "Revenue", "Cost_Of_Goods",'Promotions','Staffing'
]

for table in tables:
    cur.execute(f"SELECT COUNT(*) FROM {table}")
    count = cur.fetchone()[0]
    print(f"{table}: {count} rows")

cur.close()
conn.close()

Employees: 1000 rows
Human_Resources: 1000 rows
Customer_Loyalty: 1000 rows
Vendors: 200 rows
Stores: 50 rows
Inventory: 2000 rows
Delivery_Tracking: 500 rows
Sales_Management: 1000 rows
Returns_Refunds: 300 rows
Operations_Billing: 300 rows
Asset_Management: 300 rows
Vehicle_Management: 300 rows
Security_Incidents: 300 rows
Real_Estate_Development: 30 rows
Transactions: 1000 rows
Revenue: 300 rows
Cost_Of_Goods: 300 rows
Promotions: 300 rows
Staffing: 300 rows


In [30]:

conn = psycopg2.connect(
    host="localhost",
    port='5432',
    dbname="abcmart-test",
    user="postgres",
    password="1234"             
)

cur = conn.cursor()

drop_sql = """
DROP TABLE IF EXISTS
    Cost_Of_Goods,
    Revenue,
    Transactions,
    Real_Estate_Development,
    Security_Incidents,
    Vehicle_Management,
    Asset_Management,
    Operations_Billing,
    Returns_Refunds,
    Sales_Management,
    Delivery_Tracking,
    Inventory,
    Stores,
    Vendors,
    Customer_Loyalty,
    Human_Resources,
    Employees,
    Promotions,
    Staffing
CASCADE;
"""

cur.execute(drop_sql)
conn.commit()
cur.close()
conn.close()

print("All tables dropped.")

All tables dropped.
